- references
    - https://lmsys.org/blog/2025-12-03-miles-fsdp/
    - https://github.com/zhaochenyang20/Awesome-ML-SYS-Tutorial/blob/main/rlhf/sys-design/readme-2.md

- DDP, DeepSpeed, FSDP
    - 本质上首先都是 DP
    - FSDP (Fully Sharded Data Parallel) inherits the design philosophy of DeepSpeed ZeRO Stage 3 and can be seen as a powerful optimization of traditional DDP (Distributed Data Parallel).
- TP: Tensor parallelism **shards model parameters** across multiple GPUs within each model layer. This is the most common strategy for large model inference within a single node.
    - 同一 tp_group 里做 all-reduce / all-gather。
- PP: Pipeline parallelism **distributes model layers** across multiple GPUs. Each GPU processes different parts of the model in sequence.
    - When you've already maxed out efficient tensor parallelism but need to distribute the model further, or across nodes
    - For very deep and narrow models where layer distribution is more efficient than tensor sharding
    - 同一 pp_group 做点对点 send/recv

### DeepSpeed

Zero Stage 3 加载时将模型参数进行切片存储到不同的GPU上，每个GPU只保留参数的1/N。计算时，每个GPU跑不同的数据，然后GPU之间进行参数通信，保证每个GPU下的batch都能通过模型全部参数，而不是局部参数。（主要利用 `all-gather` 收集参数，`reduce-scatter` 规约计算）。
- 每个 GPU 依然在处理不同的数据 mini-batch；
- 每个 GPU 逻辑上拥有完整模型，只是参数暂时存储在不同 GPU 上；
- 计算时，通过参数分片广播机制，当前 GPU 拿到自己需要的参数块进行 forward/backward；
- 所以它依然属于 数据并行范畴，只是通过“分布式状态管理”节省显存。

### FSDP

In traditional DDP, each GPU maintains a complete copy of model weights, gradients, and optimizer states (Replication), synchronizing gradients via all-reduce. In FSDP, we shift to a Sharding mode: all the aforementioned data is sharded and distributed across different GPU ranks.
- Forward Propagation: When a **layer** needs to be calculated, full parameters are temporarily collected via `all-gather` and released immediately after calculation.
- Backward Propagation: After gradient calculation is complete, `reduce-scatter` is performed immediately to synchronize and shard, then the full gradients are released.

- fsdp1 -> fsdp2

- FSDP 的 process_group=dp_group（只在 DP 维上通信）
- TP 层内部算子用 tp_group
- PP 只在相邻 pipeline 段之间点对点发/收